In [1]:
pip install transformers datasets torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import TrainingArguments, Trainer, AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from datasets import Dataset
import torch
import pandas as pd

/home/avlarionov/.conda/envs/myenv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('bank-full.csv', sep=';')
df.head(4)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no


In [4]:
# Преобразование данных в текст
def concatenate_text_bank(x):
    # Преобразуем бинарные значения в текст
    dft = 'has a credit in default' if x['default'] == 'yes' else 'has no credit in default'
    hsg = 'has housing loan' if x['housing'] == 'yes' else "doesn't have housing loan"
    ln = 'has a personal loan' if x['loan'] == 'yes' else 'has no personal loan'

    # Формируем текст описания
    full_text = (
        f"This customer is {x['age']} years old, "
        f"with {x['job']} occupation, "
        f"is {x['marital']}, "
        f"with {x['education']} level of education, "
        f"{dft}, "
        f"with average yearly balance of {x['balance']} euros, "
        f"{hsg}, "
        f"{ln}, "
        f"contacted via {x['contact']} type of contact, "
        f"on {x['day']} day of {x['month']}, "
        f"with the last call duration of {x['duration']} seconds, "
        f"{x['campaign']} times of call in current marketing campaign, "
        f"with {x['pdays']} days of pass between contacts, "
        f"{x['previous']} times of contacts in the previous campaign, "
        f"and last poutcome is {x['poutcome']}."
    )
    return full_text

In [5]:
df['text'] = df.apply(lambda x: concatenate_text_bank(x), axis=1)
df['text'].iloc[3]

'This customer is 47 years old, with blue-collar occupation, is married, with unknown level of education, has no credit in default, with average yearly balance of 1506 euros, has housing loan, has no personal loan, contacted via unknown type of contact, on 5 day of may, with the last call duration of 92 seconds, 1 times of call in current marketing campaign, with -1 days of pass between contacts, 0 times of contacts in the previous campaign, and last poutcome is unknown.'

In [6]:
# Преобразование в формат Dataset
dataset = Dataset.from_pandas(df[['text', 'y']])
dataset

Dataset({
    features: ['text', 'y'],
    num_rows: 45211
})

In [7]:
# Загрузка модели и токенизатора
model = AutoModelForSeq2SeqLM.from_pretrained("bigScience/T0_3B")
tokenizer = AutoTokenizer.from_pretrained("bigScience/T0_3B")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [8]:
# Проверяем, доступен ли GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Перемещение модели на GPU
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 2048)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 2048)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=2048, out_features=2048, bias=False)
              (k): Linear(in_features=2048, out_features=2048, bias=False)
              (v): Linear(in_features=2048, out_features=2048, bias=False)
              (o): Linear(in_features=2048, out_features=2048, bias=False)
              (relative_attention_bias): Embedding(32, 32)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=2048, out_features=5120, bias=False)
              (wi_1): Linear(in_features=2048, out_features=5120, bias=False)
       

In [9]:
def tokenize_function(examples):
    inputs = tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    labels = tokenizer(examples['y'], padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    inputs['labels'] = labels['input_ids']
    return inputs

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 45211/45211 [00:14<00:00, 3080.27 examples/s]


In [10]:
tokenized_dataset

Dataset({
    features: ['text', 'y', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 45211
})

In [11]:
# Разделение данных на train и test
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

train_data = split_dataset['train']
test_data = split_dataset['test']

In [12]:
from collections import Counter

y_values = train_data['y']
if isinstance(y_values, list):
    print(Counter(y_values))

Counter({'no': 31966, 'yes': 4202})


In [13]:
y_values_test = test_data['y']
if isinstance(y_values_test, list):
    print(Counter(y_values_test))

Counter({'no': 7956, 'yes': 1087})


In [14]:
# Удаляем ненужные колонки
train_data = train_data.remove_columns(['text', 'y'])
test_data = test_data.remove_columns(['text', 'y'])

In [15]:
train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [16]:
# Проверка формы input_ids и attention_mask
print("Форма input_ids:", train_data[0]['input_ids'].shape)
print("Форма attention_mask:", train_data[0]['attention_mask'].shape)
print("Форма labels:", train_data[0]['labels'].shape)


Форма input_ids: torch.Size([128])
Форма attention_mask: torch.Size([128])
Форма labels: torch.Size([128])


In [17]:
train_data['attention_mask'][0]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])

In [18]:
train_data['input_ids'][0]

tensor([  100,   884,    19,  3538,   203,   625,     6,    28, 10473,     5,
        13792,     6,    19,   712,     6,    28,  6980,   593,    13,  1073,
            6,    65,   150,   998,    16,  4647,     6,    28,  1348,     3,
        22286,  2109,    13,     3,    18,  2668, 10186,     6,   744,    31,
           17,    43,  3499,  2289,     6,    65,   150,   525,  2289,     6,
            3, 12655,  1009,     3, 10791,   686,    13,   574,     6,    30,
         2838,   239,    13,     3,  7066,     6,    28,     8,   336,   580,
         8659,    13,     3, 15003,  3978,     6,   209,   648,    13,   580,
           16,   750,  1070,  2066,     6,    28,     3, 21594,   477,    13,
         1903,   344, 11463,     6,   305,   648,    13, 11463,    16,     8,
         1767,  2066,     6,    11,   336, 10964,    17,   287,    15,    19,
          119,     5,     1,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0])

In [19]:
train_data['labels'][0]

tensor([150,   1,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0])

In [20]:
# Заморозка всех слоев в энкодере
for param in model.encoder.parameters():
    param.requires_grad = False

# Заморозка всех слоев в декодере, кроме 23-го блока
for i, layer in enumerate(model.decoder.block):
    if i != 23:  # Указание на 23-й слой (индекс 23)
        for param in layer.parameters():
            param.requires_grad = False

# Проверка заморозки
for name, param in model.named_parameters():
    print(f"{name}: requires_grad = {param.requires_grad}")

shared.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.q.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.k.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.v.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.o.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight: requires_grad = False
encoder.block.0.layer.0.layer_norm.weight: requires_grad = False
encoder.block.0.layer.1.DenseReluDense.wi_0.weight: requires_grad = False
encoder.block.0.layer.1.DenseReluDense.wi_1.weight: requires_grad = False
encoder.block.0.layer.1.DenseReluDense.wo.weight: requires_grad = False
encoder.block.0.layer.1.layer_norm.weight: requires_grad = False
encoder.block.1.layer.0.SelfAttention.q.weight: requires_grad = False
encoder.block.1.layer.0.SelfAttention.k.weight: requires_grad = False
encoder.block.1.layer.0.SelfAttention.v.weight: requires_grad = False
encoder.block.1.layer.0.SelfAtt

In [21]:
training_args = TrainingArguments(
    output_dir='./results',
    save_total_limit=2,
    run_name="my_t0_experiment",
    report_to="none",
    eval_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [22]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.002300,0.002037
2,0.002100,0.001889
3,0.002100,0.001921
4,0.002100,0.001817
5,0.002000,0.001801
6,0.001900,0.001774
7,0.001800,0.001742
8,0.001800,0.001736
9,0.001900,0.001708
10,0.001700,0.001704


TrainOutput(global_step=22610, training_loss=0.032713617097746214, metrics={'train_runtime': 23310.5831, 'train_samples_per_second': 15.516, 'train_steps_per_second': 0.97, 'total_flos': 7.733004018175181e+17, 'train_loss': 0.032713617097746214, 'epoch': 10.0})

In [23]:
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

In [24]:
# Функция для декодирования меток из тензоров в текст
def decode_labels(labels):
    decoded_labels = []
    for label in labels:
        decoded_label = tokenizer.decode(label, skip_special_tokens=True).strip()
        decoded_labels.append(decoded_label)
    return decoded_labels

In [25]:
# Функция для предсказания
def predict_tokens(input_ids):
    device = next(model.parameters()).device
    # Создаем промпт для модели
    prompt = "Does this client subscribe to a term deposit? Yes or no?"

    # Токенизируем промпт
    prompt_ids = tokenizer(prompt, return_tensors="pt", max_length=128, truncation=True).to(device)

    # Объединяем input_ids и prompt_ids
    input_ids = input_ids.to("cuda")
    combined_input_ids = torch.cat([input_ids, prompt_ids['input_ids']], dim=-1)

    # Генерируем ответ модели
    with torch.no_grad():
        output_ids = model.generate(input_ids=combined_input_ids, max_length=64)

    # Декодируем ответ модели в текст
    predicted_class_text = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    predicted_class = 'yes' if predicted_class_text.lower() == 'yes' else 'no'

    # Возвращаем класс, output_ids и предсказанный текст
    return predicted_class, predicted_class_text

In [26]:
# Функция для получения предсказаний
def get_predictions(test_data, sample_range=None):
    predictions = []
    true_labels = []
    predicted_texts = []  # Список для хранения предсказанных текстов

    # Определяем диапазон обработки
    if sample_range:
        start_idx, end_idx = sample_range
    else:
        start_idx, end_idx = 0, len(test_data)

    # Декодируем истинные метки
    true_labels_text = decode_labels(test_data['labels'][start_idx:end_idx])

    for i in range(start_idx, end_idx):
        example = test_data[i]

        # Получаем предсказание
        predicted_class, predicted_text = predict_tokens(example['input_ids'].unsqueeze(0))

        # Сохраняем предсказанный текст
        predicted_texts.append(predicted_text)

        # Преобразуем метки в числовой формат
        true_labels.append(1 if true_labels_text[i - start_idx] == 'yes' else 0)
        predictions.append(1 if predicted_class == 'yes' else 0)

        # Выводим предсказанный текст для каждого примера (только для выборки)
        if sample_range:
            print(f"Example {i + 1}:")
            print(f"  True label: {true_labels_text[i - start_idx]}")
            print(f"  Predicted text: {predicted_text}")
            print(f"  Predicted class: {predicted_class}")
            print("-" * 40)

    return true_labels, predictions, predicted_texts

In [27]:
# Функция для вычисления метрик
def calculate_metrics(true_labels, predictions):
    roc_auc = roc_auc_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions)
    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)

    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")

In [28]:
true_labels_full, predictions_full, _ = get_predictions(test_data)

# Вычисляем метрики для полного набора
calculate_metrics(true_labels_full, predictions_full)

ROC-AUC: 0.7189
F1 Score: 0.5377
Accuracy: 0.9009
Precision: 0.6122
Recall: 0.4793


In [29]:
# Пример использования
start_idx = 200  # Начальный индекс выборки
end_idx = 230   # Конечный индекс выборки
true_labels_sample, predictions_sample, _ = get_predictions(test_data, sample_range=(start_idx, end_idx))

Example 201:
  True label: yes
  Predicted text: yes
  Predicted class: yes
----------------------------------------
Example 202:
  True label: yes
  Predicted text: no
  Predicted class: no
----------------------------------------
Example 203:
  True label: no
  Predicted text: no
  Predicted class: no
----------------------------------------
Example 204:
  True label: no
  Predicted text: no
  Predicted class: no
----------------------------------------
Example 205:
  True label: no
  Predicted text: no
  Predicted class: no
----------------------------------------
Example 206:
  True label: no
  Predicted text: no
  Predicted class: no
----------------------------------------
Example 207:
  True label: no
  Predicted text: no
  Predicted class: no
----------------------------------------
Example 208:
  True label: no
  Predicted text: no
  Predicted class: no
----------------------------------------
Example 209:
  True label: no
  Predicted text: no
  Predicted class: no
-----------

In [30]:
prediction_counts = Counter(predictions_full)

print(prediction_counts)

Counter({0: 8192, 1: 851})


In [ ]:
# Сохранение модели и токенизатора
model.save_pretrained("./saved_model/model_10_epoch")
tokenizer.save_pretrained("./saved_model/model_10_epoch")